# Setup & Configuration

In [23]:
import os
import json
import base64
import requests
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import time
import re 
from pathlib import Path
from PIL import Image
from io import BytesIO
from openai import OpenAI

# --- API Configuration ---
OPENROUTER_API_KEY = "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"
MODEL_NAME = "openrouter/stepfun/step-3.5-flash:free" 

NGROK_URL = "https://3325-194-27-149-203.ngrok-free.app/v1"
# MODELS_TO_TEST = ["llava:7b", "qwen2.5vl:7b", "qwen3-vl:8b"]
# MODELS_TO_TEST = ["llava:7b"]
# MODELS_TO_TEST = ["qwen2.5vl:7b"]
# MODELS_TO_TEST = ["qwen3-vl:8b"]
MODELS_TO_TEST = ["qwen3-vl:4b"]
# MODELS_TO_TEST = ["qwen3-vl:4b","llava:7b"]

MAX_RETRIES = 3

# --- Dataset Paths ---
CHARTQA_PATH = Path("../DChartQA/ChartQA Dataset")
MATHVISION_PATH = Path("../DMathVision/mathvision.parquet")
SCREENQA_PATH = Path("../DScreenQA/screenqa.parquet")
TURTLEBENCH_PATH = Path("../DTurtleBench")
MTVQA_PATH = Path("../DMTVQA/data/mtvqa_AR.parquet")
PEARL_PATH = Path("../DPEARL/data/pearl.parquet")

# --- Standard Client ---
client = OpenAI(
    base_url=NGROK_URL,
    api_key="ollama",
    timeout=180.0 
)

# MathVision Evaluation

### Setup

In [ ]:
# --- Data Loading ---
dataset_path = MATHVISION_PATH

if dataset_path.exists():
    df = pd.read_parquet(dataset_path)
    print(f"✅ Loaded {len(df)} rows.")
else:
    print(f"❌ Dataset not found at {dataset_path}. Please check the path.")

✅ Loaded 3040 rows.


### Openrouter

In [ ]:
def encode_image_from_bytes(byte_data):
    """Converts raw bytes to a base64 encoded string."""
    try:
        base64_str = base64.b64encode(byte_data).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None

def query_vision_model(prompt, base64_image, max_retries=MAX_RETRIES):
    """Sends the image and prompt to the VLM, forcing a short answer and tracking execution time."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    strict_prompt = f"""{prompt}

You are an expert mathematician and visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
2. If the answer is a letter matching a multiple choice option (A, B, C, D), simply output the letter.
3. Output equations or numbers directly.
"""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": strict_prompt},
                    {"type": "image_url", "image_url": {"url": base64_image}}
                ]
            }
        ]
    }
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120
            )
            response.raise_for_status()
            result_text = response.json()['choices'][0]['message']['content'].strip()
            duration = time.time() - start_time
            return result_text, duration
        except requests.exceptions.Timeout:
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: Request timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_math(ans, gt):
    """Evaluates if the model answer matches the ground truth."""
    ans_clean = str(ans).lower().replace(" ", "")
    gt_clean = str(gt).lower().replace(" ", "")
    
    # Direct exact match
    if ans_clean == gt_clean:
        return True
    
    # Substring match (in case the model is slightly verbose despite instructions)
    if gt_clean in ans_clean:
        return True
        
    # Fallback structure
    return False

def run_evaluation(df, limit=None):
    print(f"🚀 Starting MathVision inference using {MODEL_NAME}...")
    
    safe_model_name = MODEL_NAME.replace("/", "_").replace(":", "_")
    
    # Create output directory if it does not exist
    results_dir = Path("MathVisionResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mathvision_{safe_model_name}_results.csv"
    
    processed_tasks = set()
    file_exists = output_csv.is_file()
    
    if file_exists:
        try:
            existing_df = pd.read_csv(output_csv)
            if not existing_df.empty and 'id' in existing_df.columns:
                processed_tasks = set(existing_df['id'].astype(str))
                print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
            
    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            break
            
        task_id = str(row['id'])
        if task_id in processed_tasks:
            continue
            
        level = str(row.get('level', 'N/A'))
        subject = str(row.get('subject', 'N/A'))
        image_name = str(row.get('image', 'N/A'))
        query = str(row['question'])
        ground_truth = str(row['answer']).strip()
        
        print(f"\nProcessing Task: {task_id} | Subject: {subject} | Lvl: {level}")
        
        img_data = None
        base64_img = None
        
        if 'decoded_image' in row and isinstance(row['decoded_image'], dict):
            img_data = row['decoded_image'].get('bytes')
            
        if img_data:
            base64_img = encode_image_from_bytes(img_data)
        
        if not base64_img:
            print("   ⚠️ Skipping due to image parsing error.")
            continue
            
        model_answer, duration = query_vision_model(query, base64_img)
        evaluation_passed = evaluate_math(model_answer, ground_truth)
        
         # Save to CSV
        result_df = pd.DataFrame([{
            'id': task_id,
            'level': level,
            'subject': subject,
            'image' : image_name,
            'question': query,
            'ground_truth': ground_truth,
            'model_answer': model_answer,
            'evaluation_passed': evaluation_passed,
            'run_time': round(duration, 3),
        }])
        
        result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
        file_exists = True
        
        processed_tasks.add(task_id)
        count += 1
        
        eval_mark = "✅" if evaluation_passed else "❌"
        print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
        time.sleep(3) # API rate limit pause

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_evaluation(df, limit=1)
else:
    print("❌ ‘df’ memory object missing. Please run Cell 1 first to load the dataset.")


### Ngrok

In [11]:
# Standardize Ngrok connection
client = OpenAI(
    base_url=NGROK_URL,
    api_key="sk-no-key-required"
)

def encode_image_from_bytes(byte_data):
    """Converts raw bytes to a base64 encoded string."""
    try:
        base64_str = base64.b64encode(byte_data).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None

def query_vision_model(prompt, base64_image, model_name, max_retries=MAX_RETRIES):
    """Sends the image and prompt directly via the OpenAI package to Ngrok backend."""
    strict_prompt = f"""{prompt}

You are an expert mathematician and visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
2. If the answer is a letter matching a multiple choice option (A, B, C, D), simply output the letter.
3. Output equations or numbers directly.
"""
    
    messages = [
         {
            "role": "user",
            "content": [
                {"type": "text", "text": strict_prompt},
                {"type": "image_url", "image_url": {"url": base64_image}}
            ]
        }
    ]
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                stream=False,
                temperature=0.1
            )
            result_text = response.choices[0].message.content.strip()
            duration = time.time() - start_time
            return result_text, duration
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_math(ans, gt):
    """Evaluates if the model answer matches the ground truth."""
    ans_clean = str(ans).lower().replace(" ", "")
    gt_clean = str(gt).lower().replace(" ", "")
    
    if ans_clean == gt_clean:
        return True
    if gt_clean in ans_clean:
        return True
        
    return False

def run_evaluation(df, limit=None):
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Starting MathVision NGROK inference using {model_name}...")
        
        safe_model_name = model_name.replace("/", "_").replace(":", "_")
        
        results_dir = Path("MathVisionResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"mathvision_ngrok_{safe_model_name}_results.csv"
        
        processed_tasks = set()
        file_exists = output_csv.is_file()
        
        if file_exists:
            try:
                existing_df = pd.read_csv(output_csv)
                if not existing_df.empty and 'id' in existing_df.columns:
                    processed_tasks = set(existing_df['id'].astype(str))
                    print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
            except Exception as e:
                print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
                
        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break
                
            task_id = str(row['id'])
            if task_id in processed_tasks:
                continue
                
            level = str(row.get('level', 'N/A'))
            subject = str(row.get('subject', 'N/A'))
            image_name = str(row.get('image', 'N/A'))
            query = str(row['question'])
            ground_truth = str(row['answer']).strip()
            
            print(f"\nProcessing Task: {task_id} | Subject: {subject} | Lvl: {level}")
            
            img_data = None
            base64_img = None
            
            if 'decoded_image' in row and isinstance(row['decoded_image'], dict):
                img_data = row['decoded_image'].get('bytes')
                
            if img_data:
                base64_img = encode_image_from_bytes(img_data)
            
            if not base64_img:
                print("   ⚠️ Skipping due to image parsing error.")
                continue
                
            model_answer, duration = query_vision_model(query, base64_img, model_name)
            evaluation_passed = evaluate_math(model_answer, ground_truth)
            
            # Save to CSV
            result_df = pd.DataFrame([{
                'id': task_id,
                'level': level,
                'subject': subject,
                'image' : image_name,
                'question': query,
                'ground_truth': ground_truth,
                'model_answer': model_answer,
                'evaluation_passed': evaluation_passed,
                'run_time': round(duration, 3),
            }])
            
            result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
            file_exists = True
            
            processed_tasks.add(task_id)
            count += 1
            
            eval_mark = "✅" if evaluation_passed else "❌"
            print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
            time.sleep(3) # API rate limit pause

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_evaluation(df, limit=2)
else:
    print("❌ ‘df’ memory object missing. Please run Cell 1 first to load the dataset.")



🚀 Starting MathVision NGROK inference using qwen3-vl:8b...

Processing Task: 1 | Subject: arithmetic | Lvl: 2
   ✅ Evaluation | Time: 15.36s | Reply: '60'

Processing Task: 2 | Subject: arithmetic | Lvl: 2
   ✅ Evaluation | Time: 7.07s | Reply: 'A'
🎯 Limit of 2 reached for qwen3-vl:8b.

🚀 Starting MathVision NGROK inference using llava:7b...

Processing Task: 1 | Subject: arithmetic | Lvl: 2
   ❌ Evaluation | Time: 10.07s | Reply: '0'

Processing Task: 2 | Subject: arithmetic | Lvl: 2
   ✅ Evaluation | Time: 1.90s | Reply: 'A'
🎯 Limit of 2 reached for llava:7b.


# ChartQA Evaluation

### Setup

In [75]:
def load_chart_qa_split(dataset_path, split_name, type="augmented"):
    """Loads ChartQA data for a specific split (train, val, test)."""
    json_file = dataset_path / split_name / f"{split_name}_{type}.json"
    image_dir = dataset_path / split_name / "png"
    
    if not json_file.exists():
        raise FileNotFoundError(f"Cannot find dataset at: {json_file}")

    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    df = pd.DataFrame(data)
    # Standardize image path mapping
    df["image_path"] = df["imgname"].apply(lambda x: str(image_dir / x))
    return df

# --- Data Loading ---
try:
    chartqa_df = load_chart_qa_split(CHARTQA_PATH, "train", type="augmented")
    print(f"✅ Loaded {len(chartqa_df)} augmented-authored ChartQA rows from train split.")
except Exception as e:
    print(f"❌ Error loading ChartQA: {e}")

✅ Loaded 20901 augmented-authored ChartQA rows from train split.


### Openrouter

In [ ]:
# --- 1. Data Loading & Utility Functions ---
def load_chart_qa_split(dataset_path, split_name, type="augmented"):
    json_file = os.path.join(dataset_path, split_name, f"{split_name}_{type}.json")
    image_dir = os.path.join(dataset_path, split_name, "png")
    if not os.path.exists(json_file):
        raise FileNotFoundError(f"Cannot find dataset at: {json_file}")
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df["image_path"] = df["imgname"].apply(lambda x: os.path.join(image_dir, x))
    return df

def encode_and_resize_image(image_path):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            base64_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
            return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None

# --- 2. API Queries ---
def query_vision_model_openrouter(prompt, base64_image, max_retries=MAX_RETRIES):
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    strict_prompt = f"""{prompt}

You are a strict data extraction API analyzing a chart. Provide the exact answer to the user's question using only the visual data.

STRICT FORMATTING RULES:
1. RAW DATA ONLY: Output the final answer and nothing else. Zero conversational text, zero reasoning, and zero introductory filler (Never write "The answer is" or "Based on the chart").
2. NUMBERS: Output the exact digits. Include units, currencies, or percentages only if they are the direct answer (e.g., "45.2%" or "$120"). Do not spell out numbers as words.
3. BOOLEANS: If the question implies a true/false or yes/no answer, output strictly "Yes" or "No".
4. LISTS: If the answer requires multiple chart labels, separate them strictly with a comma and a space (e.g., "France, Germany, Italy").
5. EXACT MATCH: Copy axis labels, legend items, or categories exactly as they appear in the chart image.
"""
    payload = {
        "model": MODEL_NAME,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": strict_prompt},
                {"type": "image_url", "image_url": {"url": base64_image}}
            ]
        }],
        "temperature": 0.1
    }
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120
            )
            response.raise_for_status()
            result_text = response.json()['choices'][0]['message']['content'].strip()
            return result_text, time.time() - start_time
        except Exception as e:
            print(f"   ❌ API Attempt {attempt + 1}: {e}")
            time.sleep(5 * (attempt + 1))
    return "ERROR_MAX_RETRIES_REACHED", time.time() - start_time

# --- 3. Evaluation ---
def evaluate_chartqa(ans, gt):
    """Evaluates if the model answer matches the ground truth (relaxed)."""
    ans_clean = str(ans).lower().strip()
    valid_labels = gt if isinstance(gt, list) else [gt]
    
    for label in valid_labels:
        if str(label).lower() in ans_clean:
            return True
    return False

# --- 4. Processing Pipeline ---
def process_chartqa_openrouter(df, limit=None):
    print(f"🚀 Starting ChartQA OpenRouter inference using {MODEL_NAME}...")
    
    safe_model_name = MODEL_NAME.replace("/", "_").replace(":", "_")
    results_dir = Path("ChartQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"chartqa_openrouter_{safe_model_name}_results.csv"
    
    processed_tasks = set()
    file_exists = output_csv.is_file()
    
    if file_exists:
        try:
            existing_df = pd.read_csv(output_csv)
            if not existing_df.empty and 'imgname' in existing_df.columns and 'question' in existing_df.columns:
                # Combine imgname and question to create a unique task key
                processed_tasks = set(existing_df['imgname'].astype(str) + "_" + existing_df['question'].astype(str))
                print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
            
    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached for {MODEL_NAME}.")
            break
            
        imgname = str(row['imgname'])
        query = str(row['query'])
        task_key = imgname + "_" + query
        
        if task_key in processed_tasks:
            continue
            
        image_path = row['image_path']
        ground_truth = row['label']
        
        print(f"\nProcessing Image: {imgname} | Query: {query}")
        
        base64_img = encode_and_resize_image(image_path)
        if not base64_img:
            print("   ⚠️ Skipping due to image parsing error.")
            continue
            
        model_answer, duration = query_vision_model_openrouter(query, base64_img)
        evaluation_passed = evaluate_chartqa(model_answer, ground_truth)
        
        # Save to CSV
        result_df = pd.DataFrame([{ 
            'imgname': imgname,
            'image_path': image_path,
            'question': query,
            'ground_truth': str(ground_truth),
            'model_answer': model_answer,
            'evaluation_passed': evaluation_passed,
            'run_time': round(duration, 3),
        }])
        
        result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
        file_exists = True
        
        processed_tasks.add(task_key)
        count += 1
        
        eval_mark = "✅" if evaluation_passed else "❌"
        print(f"   {eval_mark} Match | Time: {duration:.2f}s | Reply: '{model_answer}' | GT: '{ground_truth}'")
        time.sleep(3) # API rate limit pause

# --- Execution ---
if __name__ == "__main__":
    
    try:
        if chartqa_df is not None and not chartqa_df.empty:
            print(f"✅ Loaded {len(chartqa_df)} questions.")
            # Set limit to test just the first few, or remove limit=2 to run everything
            process_chartqa_openrouter(chartqa_df, limit=2) 
        else:
            print("⚠️ Dataset empty.")
    except Exception as e:
        print(f"❌ Fatal Error: {e}")

### Ngrok

In [ ]:
def encode_image(image_path):
    """Encodes image to base64."""
    try:
        with open(image_path, "rb") as image_file:
            base64_str = base64.b64encode(image_file.read()).decode('utf-8')
            return f"data:image/png;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding image {image_path}: {e}")
        return None

def query_vision_model(prompt, base64_image, model_name, max_retries=MAX_RETRIES):
    """Sends image and prompt to Ngrok backend."""
    strict_prompt = f"""{prompt}

You are a strict data extraction API analyzing a chart. Provide the exact answer to the text question input using only the visual data.

STRICT FORMATTING RULES:
1. RAW DATA ONLY: Output the final answer and nothing else. Zero conversational text, zero reasoning, and zero introductory filler (Never write "The answer is" or "Based on the chart").
2. NUMBERS: Output the exact digits. Include units, currencies, or percentages only if they are the direct answer (e.g., "45.2%" or "$120"). Do not spell out numbers as words.
3. BOOLEANS: If the question implies a true/false or yes/no answer, output strictly "Yes" or "No".
4. LISTS: If the answer requires multiple chart labels, separate them strictly with a comma and a space (e.g., "France, Germany, Italy").
5. EXACT MATCH: Copy axis labels, legend items, or categories exactly as they appear in the chart image.
"""
    
    messages = [
         {
            "role": "user",
            "content": [
                {"type": "text", "text": strict_prompt},
                {"type": "image_url", "image_url": {"url": base64_image}}
            ]
        }
    ]
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                stream=False,
                temperature=0.1
            )
            result_text = response.choices[0].message.content.strip()
            duration = time.time() - start_time
            return result_text, duration
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_chartqa(ans, gt):
    """Evaluates if the model answer matches the ground truth (relaxed)."""
    ans_clean = str(ans).lower().strip()
    # ChartQA sometimes has multiple valid answers in a list
    valid_labels = gt if isinstance(gt, list) else [gt]
    
    for label in valid_labels:
        if str(label).lower() in ans_clean:
            return True
    return False

def run_evaluation(df, limit=None):
    for model_name in MODELS_TO_TEST:
        print(f"🚀 Starting ChartQA NGROK inference using {model_name}...")
        
        safe_model_name = model_name.replace("/", "_").replace(":", "_")
        
        # Standardized results directory
        results_dir = Path("ChartQAResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"chartqa_ngrok_{safe_model_name}_results.csv"
        
        processed_tasks = set()
        file_exists = output_csv.is_file()
        
        if file_exists:
            try:
                existing_df = pd.read_csv(output_csv)
                # Note: Entry index is used for unique ID in ChartQA as provided by dataframe index
                if not existing_df.empty and 'id' in existing_df.columns:
                    processed_tasks = set(existing_df['id'].astype(str))
                    print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
            except Exception as e:
                print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
                
        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break
                
            task_id = str(index)
            if task_id in processed_tasks:
                continue
                
            image_path = row['image_path']
            image_name = Path(image_path).name
            query = str(row['query'])
            ground_truth = row['label']
            
            print(f"Processing Task: {task_id} | Image: {image_name}")
            
            base64_img = encode_image(image_path)
            if not base64_img:
                print("   ⚠️ Skipping due to image parsing error.")
                continue
                
            model_answer, duration = query_vision_model(query, base64_img, model_name)
            evaluation_passed = evaluate_chartqa(model_answer, ground_truth)
            
            # Save to CSV
            result_df = pd.DataFrame([{ 
                'id': task_id,
                'image' : str(Path(image_path).relative_to(CHARTQA_PATH)),
                'question': query,
                'ground_truth': str(ground_truth),
                'model_answer': model_answer,
                'evaluation_passed': evaluation_passed,
                'run_time': round(duration, 3),
            }])
            
            result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
            file_exists = True
            
            processed_tasks.add(task_id)
            count += 1
            
            eval_mark = "✅" if evaluation_passed else "❌"
            print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
            time.sleep(3) # API rate limit pause

if 'chartqa_df' in globals():
    # Change limit to None to run full evaluation
    run_evaluation(chartqa_df, limit=2)
else:
    print("❌ ‘chartqa_df’ memory object missing. Please run the setup cell first.")

🚀 Starting ChartQA NGROK inference using qwen3-vl:4b...
🔄 Found existing CSV! Resuming... Skipping 4 already processed tasks.
Processing Task: 4 | Image: multi_col_40810.png
   ✅ Evaluation | Time: 61.35s | Reply: 'Lombardy'
Processing Task: 5 | Image: two_col_62773.png
   ✅ Evaluation | Time: 16.85s | Reply: '76.4 million euros'
🎯 Limit of 2 reached for qwen3-vl:4b.
🚀 Starting ChartQA NGROK inference using llava:7b...
🔄 Found existing CSV! Resuming... Skipping 4 already processed tasks.
Processing Task: 4 | Image: multi_col_40810.png
   ❌ Evaluation | Time: 304.60s | Reply: '1000000
200000
300000
400000
500000
600000
700000
800000
900000
1000000

Italy'
Processing Task: 5 | Image: two_col_62773.png
   ❌ Evaluation | Time: 2.37s | Reply: '100000000'
🎯 Limit of 2 reached for llava:7b.


# TurtleBench Evaluation

### Openrouter

In [80]:
# --- 1. Image Encoding ---
def encode_and_resize_image(image_path):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            base64_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
            return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None

# --- 2. API Queries ---
def query_vision_model(prompt, base64_image, max_retries=3):
    """Sends the image and prompt to generate the Turtle code, tracking response time."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    strict_prompt = f"""{prompt}

INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution. 
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": strict_prompt},
                    {"type": "image_url", "image_url": {"url": base64_image}}
                ]
            }
        ]
    }
    
    for attempt in range(max_retries):
        try:
            start_time = time.time()
            response = requests.post(
                url="[https://openrouter.ai/api/v1/chat/completions](https://openrouter.ai/api/v1/chat/completions)",
                headers=headers,
                json=payload,
                timeout=120 
            )
            response.raise_for_status() 
            end_time = time.time()
            generation_duration = round(end_time - start_time, 2)
            result_text = response.json()['choices'][0]['message']['content']
            return result_text, generation_duration
        except requests.exceptions.Timeout:
            print(f"   ⏳ Attempt {attempt + 1}/{max_retries}: Generation Request timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: Generation API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR_MAX_RETRIES_REACHED", 0.0

def evaluate_code_equivalence(model_code, ground_truth_code, max_retries=3):
    """Sends the generated code and truth code back to the API for evaluation."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    eval_prompt = f"""Compare the following two Python Turtle scripts. 
Analyze their logic and determine if they will generate the exact same visual shape/output on the screen.
Answer ONLY with 'Yes' or 'No'. Do not explain your reasoning.

Model Generated Code:
{model_code}

Ground Truth Code:
{ground_truth_code}"""

    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": eval_prompt}]
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(
                url="[https://openrouter.ai/api/v1/chat/completions](https://openrouter.ai/api/v1/chat/completions)",
                headers=headers,
                json=payload,
                timeout=120 
            )
            response.raise_for_status() 
            raw_response = response.json()['choices'][0]['message']['content'].strip()
            
            if "yes" in raw_response.lower():
                return True
            elif "no" in raw_response.lower():
                return False
            else:
                return raw_response 
                
        except requests.exceptions.Timeout:
            print(f"   ⏳ Eval Attempt {attempt + 1}/{max_retries}: Evaluation Request timed out.")
        except requests.exceptions.RequestException as e:
            print(f"   ❌ Eval Attempt {attempt + 1}/{max_retries}: Evaluation API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR"

# --- 3. Dataset Loading ---
def load_turtle_bench_from_folders(dataset_path):
    tasks_dir = dataset_path / "Tasks"
    if not tasks_dir.exists():
        print(f"❌ ERROR: Could not find the Tasks folder at {tasks_dir}")
        return None

    data = []
    for task_folder in tasks_dir.iterdir():
        if not task_folder.is_dir():
            continue
            
        task_id = task_folder.name
        text_dir = task_folder / "QA" / "text"
        code_dir = task_folder / "QA" / "code"
        image_path = task_folder / "image" / f"{task_id}.png"
        
        if text_dir.exists():
            for query_file in text_dir.glob("q*.txt"):
                question_id = query_file.stem
                
                with open(query_file, 'r', encoding='utf-8') as f:
                    query_text = f.read().strip()
                
                code_file = code_dir / f"{question_id}_code.txt"
                code_text = "No code found."
                if code_file.exists():
                    with open(code_file, 'r', encoding='utf-8') as f:
                        code_text = f.read().strip()
                
                data.append({
                    'task_id': task_id,
                    'question_id': question_id,
                    'query_text': query_text,
                    'code_text': code_text,
                    'task_folder': str(task_folder),
                    'query_path': str(query_file),
                    'code_path': str(code_file),
                    'image_path': str(image_path) # Absolute image path loaded here
                })
    return pd.DataFrame(data)

def find_turtlebench_automatically():
    current_dir = Path.cwd()
    for parent in [current_dir, *current_dir.parents]:
        if parent.name.lower() == "thesis":
            target_path = parent / "DTurtleBench"
            if target_path.exists():
                return target_path
    
    fallback_path = Path("/Users/smeh/Desktop/Thesis/DTurtleBench")
    if fallback_path.exists():
        return fallback_path
    return None

# --- 4. Processing Pipeline & Evaluation ---
def process_dataset_to_csv(df):
    print(f"🚀 Starting inference on {len(df)} questions using {MODEL_NAME}...")
    
    safe_model_name = MODEL_NAME.replace("/", "_").replace(":", "_")
    results_dir = Path('TurtleBenchResults')
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"TurtleBench_{safe_model_name}_results.csv"
    
    processed_tasks = set()
    file_exists = output_csv.is_file()
    
    if file_exists:
        try:
            existing_df = pd.read_csv(output_csv)
            if not existing_df.empty and 'task_id' in existing_df.columns and 'question_id' in existing_df.columns:
                processed_tasks = set(existing_df['task_id'].astype(str) + "_" + existing_df['question_id'].astype(str))
                print(f"🔄 Found existing results! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
            
    for index, row in df.iterrows():
        task_key = str(row['task_id']) + "_" + str(row['question_id'])
        if task_key in processed_tasks:
            continue 
            
        print(f"\nProcessing Task: {row['task_id']} | Question: {row['question_id']}...")
        
        # 1. Vision Request
        # Image path retrieved from row
        base64_img = encode_and_resize_image(row['image_path'])
        if not base64_img:
            print("   ⚠️ Skipping due to image error.")
            continue
            
        model_response, generation_time = query_vision_model(row['query_text'], base64_img)
        ground_truth = row['code_text'].strip()
        
        print(f"   ⏱️ Model generated code in {generation_time} seconds.")
        
        # --- PAUSE AFTER FIRST REQUEST ---
        print("   ⏳ Waiting 2 seconds before sending evaluation request...")
        time.sleep(2)
        
        # 2. Text Evaluation Request
        print("   🔍 Sending codes to model for output equivalence check...")
        evaluation_passed = evaluate_code_equivalence(model_response, ground_truth)
        
        # --- UPDATED: Image path added to CSV DataFrame ---
        result_df = pd.DataFrame([{
            'task_id': row['task_id'],
            'question_id': row['question_id'],
            'image_path': row['image_path'],
            'query_text': row['query_text'],
            'model_response': model_response,
            'ground_truth_code': ground_truth,
            'codegeneration_time': generation_time,
            'evaluation_passed': evaluation_passed
        }])
        
        result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
        file_exists = True 
        
        processed_tasks.add(task_key)
        eval_mark = "✅" if evaluation_passed is True else "❌"
        print(f"   {eval_mark} Task complete | Saved to merged results CSV.")
        
        # --- PAUSE AFTER SECOND REQUEST ---
        print("   ⏳ Waiting 2 seconds before moving to the next task...")
        time.sleep(2)

# --- Execution ---
if __name__ == "__main__":
    dynamic_dataset_path = find_turtlebench_automatically()
    if dynamic_dataset_path:
        df = load_turtle_bench_from_folders(dynamic_dataset_path)

        if df is not None and not df.empty:
            print(f"✅ Successfully loaded {len(df)} questions.\n")
            process_dataset_to_csv(df.head(2))
        else:
            print("⚠️ No questions found.")
    else:
        print("❌ Could not find the 'DTurtleBench' folder.")

✅ Successfully loaded 260 questions.

🚀 Starting inference on 2 questions using openrouter/healer-alpha...

Processing Task: 61 | Question: q1...
   ❌ Attempt 1/3: Generation API Request failed: No connection adapters were found for '[https://openrouter.ai/api/v1/chat/completions](https://openrouter.ai/api/v1/chat/completions)'
   ❌ Attempt 2/3: Generation API Request failed: No connection adapters were found for '[https://openrouter.ai/api/v1/chat/completions](https://openrouter.ai/api/v1/chat/completions)'


KeyboardInterrupt: 

### Ngrok

In [ ]:
# --- 1. Image Encoding ---
def encode_and_resize_image(image_path):
    try:
        with Image.open(image_path) as img:
            if img.mode != 'RGB':
                img = img.convert('RGB')
            buffer = BytesIO()
            img.save(buffer, format="JPEG", quality=85)
            base64_str = base64.b64encode(buffer.getvalue()).decode('utf-8')
            return f"{base64_str}" 
    except Exception as e:
        print(f"❌ Error processing image {image_path}: {e}")
        return None

# --- 2. API Queries ---
def query_vision_model(model_name, prompt, base64_image, max_retries=3):
    """Sends the image and prompt to generate the Turtle code, tracking response time."""
    strict_prompt = f"""{prompt}

INSTRUCTIONS:
1. Write the Python Turtle graphics code to generate the requested image.
2. Output ONLY the raw Python code with no setup for execution. 
3. DO NOT wrap the code in markdown formatting (e.g., do not use ```python).
4. DO NOT output any conversational text, explanations, or greetings.
5. Assume the turtle object is named `t` and the math module is imported if needed.
"""
    
    for attempt in range(max_retries):
        try:
            start_time = time.time()
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": strict_prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                        ]
                    }
                ],
                temperature=0.1
            )
            end_time = time.time()
            generation_duration = round(end_time - start_time, 2)
            result_text = response.choices[0].message.content
            return result_text, generation_duration
            
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: Generation API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR_MAX_RETRIES_REACHED", 0.0

def evaluate_code_equivalence(model_name, model_code, ground_truth_code, max_retries=3):
    """Sends the generated code and truth code back to the API for evaluation."""
    eval_prompt = f"""Compare the following two Python Turtle scripts. 
Analyze their logic and determine if they will generate the exact same visual shape/output on the screen.
Answer ONLY with 'Yes' or 'No'. Do not explain your reasoning.

Model Generated Code:
{model_code}

Ground Truth Code:
{ground_truth_code}"""

    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=[{"role": "user", "content": eval_prompt}],
                temperature=0.1
            )
            raw_response = response.choices[0].message.content.strip()
            
            if "yes" in raw_response.lower():
                return True
            elif "no" in raw_response.lower():
                return False
            else:
                return raw_response 
                
        except Exception as e:
            print(f"   ❌ Eval Attempt {attempt + 1}/{max_retries}: Evaluation API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    return "ERROR"
    
# --- 3. Dataset Loading ---
def load_turtle_bench_from_folders(dataset_path):
    tasks_dir = Path(dataset_path) / "Tasks"
    if not tasks_dir.exists():
        print(f"❌ ERROR: Could not find the Tasks folder at {tasks_dir}")
        return None

    data = []
    for task_folder in tasks_dir.iterdir():
        if not task_folder.is_dir():
            continue
            
        task_id = task_folder.name
        text_dir = task_folder / "QA" / "text"
        code_dir = task_folder / "QA" / "code"
        image_path = task_folder / "image" / f"{task_id}.png"
        
        if text_dir.exists():
            for query_file in text_dir.glob("q*.txt"):
                question_id = query_file.stem
                
                with open(query_file, 'r', encoding='utf-8') as f:
                    query_text = f.read().strip()
                
                code_file = code_dir / f"{question_id}_code.txt"
                code_text = "No code found."
                if code_file.exists():
                    with open(code_file, 'r', encoding='utf-8') as f:
                        code_text = f.read().strip()
                
                data.append({
                    'task_id': task_id,
                    'question_id': question_id,
                    'query_text': query_text,
                    'code_text': code_text,
                    'task_folder': str(task_folder),
                    'query_path': str(query_file),
                    'code_path': str(code_file),
                    'image_path': str(image_path) # Extracted image path here
                })
    return pd.DataFrame(data)

def find_turtlebench_automatically():
    current_dir = Path.cwd()
    for parent in [current_dir, *current_dir.parents]:
        if parent.name.lower() == "thesis":
            target_path = parent / "DTurtleBench"
            if target_path.exists():
                return target_path
    
    # Fallback to absolute path
    fallback_path = Path("/Users/smeh/Desktop/Thesis/DTurtleBench")
    if fallback_path.exists():
        return fallback_path
    return None

# --- 4. Processing Pipeline & Evaluation ---
def process_dataset_to_csv(df):
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Starting inference on {len(df)} questions using {model_name}...")
        
        safe_model_name = model_name.replace("/", "_").replace(":", "_")
        results_dir = Path('TurtleBenchResults')
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"turtlebench_ngrok_{safe_model_name}_results.csv"
        
        processed_tasks = set()
        file_exists = output_csv.is_file()
        if file_exists:
            try:
                existing_df = pd.read_csv(output_csv)
                if not existing_df.empty and 'task_id' in existing_df.columns and 'question_id' in existing_df.columns:
                    processed_tasks = set(existing_df['task_id'].astype(str) + "_" + existing_df['question_id'].astype(str))
                    print(f"🔄 Resuming... Skipping {len(processed_tasks)} already processed tasks.")
            except Exception as e:
                print(f"⚠️ Resume error: {e}")
                
        for index, row in df.iterrows():
            task_key = str(row['task_id']) + "_" + str(row['question_id'])
            if task_key in processed_tasks:
                continue 
                
            print(f"\nProcessing Task: {row['task_id']} | Question: {row['question_id']}...")
            
            # Use the image path loaded from the dataset DataFrame
            base64_img = encode_and_resize_image(row['image_path'])
            if not base64_img:
                print("   ⚠️ Skipping due to image error.")
                continue
                
            model_response, generation_time = query_vision_model(model_name, row['query_text'], base64_img)
            ground_truth = row['code_text'].strip()
            
            print(f"   ⏱️ Model generated code in {generation_time} seconds.")
            time.sleep(2)
        
            print("   🔍 Sending codes to model for output equivalence check...")
            evaluation_passed = evaluate_code_equivalence(model_name, model_response, ground_truth)
       
            result_df = pd.DataFrame([{
                'task_id': row['task_id'],
                'question_id': row['question_id'],
                'image_path': row['image_path'], 
                'query_text': row['query_text'],
                'model_response': model_response,
                'ground_truth_code': ground_truth,
                'code_generation_time': generation_time,
                'evaluation_passed': evaluation_passed
            }])
            
            result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
            file_exists = True
            
            processed_tasks.add(task_key)
            eval_mark = "✅" if evaluation_passed is True else "❌"
            print(f"   {eval_mark} Task complete | Saved to merged results CSV.") 
            time.sleep(2) 

if __name__ == "__main__":
    dynamic_dataset_path = find_turtlebench_automatically()
    if dynamic_dataset_path:
        df = load_turtle_bench_from_folders(dynamic_dataset_path)
        if df is not None and not df.empty:
            print(f"✅ Loaded {len(df)} questions.")
            process_dataset_to_csv(df.head(3))
        else:
            print("⚠️ No questions found.")
    else:
        print("❌ Dataset not found.")

✅ Loaded 260 questions.

🚀 Starting inference on 3 questions using qwen3-vl:4b...

Processing Task: 61 | Question: q1...


KeyboardInterrupt: 

# ScreenQA Evaluation

### Setup

In [3]:
# --- Data Info ---
dataset_path = SCREENQA_PATH

if dataset_path.exists():
    # Read metadata only to get row count without loading full data
    metadata = pq.read_metadata(dataset_path)
    print(f"📊 ScreenQA dataset has {metadata.num_rows} rows.")
    
    df = pq.read_table(dataset_path).to_pandas()
else:
    print(f"❌ Dataset not found at {dataset_path}.")

📊 ScreenQA dataset has 86025 rows.


### Openrouter

In [13]:
def encode_image_from_bytes(byte_data):
    """Converts raw bytes to a base64 encoded string after resizing to 720p."""
    try:
        img = Image.open(BytesIO(byte_data))
        # Resize to 720p (maintaining aspect ratio)
        img.thumbnail((1280, 720))
        
        buffered = BytesIO()
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        img.save(buffered, format="JPEG", quality=85)
        base64_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding/resizing image: {e}")
        return None

def query_vision_model(prompt, base64_image, MODEL_NAME, max_retries=MAX_RETRIES):
    """Sends the image and prompt directly to the OpenRouter API."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    strict_prompt = f"""{prompt}

You are an expert visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
"""
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": strict_prompt},
                    {"type": "image_url", "image_url": {"url": base64_image}}
                ]
            }
        ],
        "temperature": 0.1
    }
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120
            )
            response.raise_for_status()
            result_text = response.json()['choices'][0]['message']['content'].strip()
            duration = time.time() - start_time
            return result_text, duration
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_screenqa(ans, gt):
    """Evaluates if the model answer matches the ground truth with high tolerance for formatting."""
    ans_str = str(ans).lower()
    gt_str = str(gt).lower()
    
    # 1. Strip currencies and commas specifically to handle number formatting (e.g., $1,000 -> 1000)
    ans_str = re.sub(r'[\$\£\€\,]', '', ans_str)
    gt_str = re.sub(r'[\$\£\€\,]', '', gt_str)
    
    # 2. Strip ALL remaining non-alphanumeric characters (spaces, periods, colons, quotes)
    ans_clean = re.sub(r'[^\w]', '', ans_str)
    gt_clean = re.sub(r'[^\w]', '', gt_str)
    
    if not ans_clean or not gt_clean:
        return False
        
    # 3. Exact match
    if ans_clean == gt_clean:
        return True
        
    # 4. Ground truth is contained within the model's (potentially verbose) answer
    if gt_clean in ans_clean:
        return True
        
    # 5. Model's answer is contained within the ground truth 
    # (Requires length >= 2 to prevent single digits like "1" matching "100" as a false positive)
    if len(ans_clean) >= 2 and ans_clean in gt_clean:
        return True
        
    return False

def run_evaluation(df, limit=None):
    print(f"\n🚀 Starting ScreenQA OPENROUTER inference using {MODEL_NAME}...")
    
    safe_MODEL_NAME = MODEL_NAME.replace("/", "_").replace(":", "_")
    
    results_dir = Path("ScreenQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    # Updated CSV name for OpenRouter
    output_csv = results_dir / f"screenqa_openrouter_{safe_MODEL_NAME}_results.csv"
    
    processed_tasks = set()
    file_exists = output_csv.is_file()
    
    if file_exists:
        try:
            existing_df = pd.read_csv(output_csv)
            if not existing_df.empty and 'id' in existing_df.columns:
                processed_tasks = set(existing_df['id'].astype(str))
                print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
        except Exception as e:
            print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
            
    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"🎯 Limit of {limit} reached for {MODEL_NAME}.")
            break
            
        task_id = str(row['screen_id'])
        if task_id in processed_tasks:
            continue
            
        image_name = str(row.get('file_name', 'N/A'))
        query = str(row['question'])
        
        # --- Safely extract 'full_answer' from the array of dicts ---
        raw_answers = row.get('answers', row.get('ground_truth', row.get('full_answer', [])))
        try:
            ground_truth = str(raw_answers[0]['full_answer']).strip()
        except (IndexError, KeyError, TypeError):
            ground_truth = str(raw_answers).strip()
        
        print(f"\nProcessing Task: {task_id} | Image: {image_name}")
        
        img_data = None
        if 'image' in row and isinstance(row['image'], dict) and 'bytes' in row['image']:
            img_data = row['image']['bytes']
        else:
            img_data = row.get('bytes') 
        
        base64_img = None
        if img_data:
            base64_img = encode_image_from_bytes(img_data)
        
        if not base64_img:
            print("   ⚠️ Skipping due to image parsing error.")
            continue
            
        model_answer, duration = query_vision_model(query, base64_img, MODEL_NAME)
        evaluation_passed = evaluate_screenqa(model_answer, ground_truth)
        
        # Save to CSV
        result_df = pd.DataFrame([{ 
            'id': task_id,
            'image' : image_name,
            'question': query,
            'ground_truth': ground_truth,
            'model_answer': model_answer,
            'evaluation_passed': evaluation_passed,
            'run_time': round(duration, 3),
        }])
        
        result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
        file_exists = True
        
        processed_tasks.add(task_id)
        count += 1
        
        eval_mark = "✅" if evaluation_passed else "❌"
        print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
        time.sleep(3) # API rate limit pause

if 'df' not in globals():
    print("🔄 Dataframe 'df' not found. Loading ScreenQA dataset...")
    # Update this path to point to your local dataset
    dataset_path = Path("path/to/your/screenqa.parquet")
    try:
        df = pq.read_table(dataset_path).to_pandas()
    except Exception as e:
        print(f"❌ Failed to load dataset: {e}")

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_evaluation(df, limit=4)
else:
    print("❌ ‘df’ memory object missing. Please run the data loading cell first.")


🚀 Starting ScreenQA OPENROUTER inference using openrouter/healer-alpha...

Processing Task: 16602 | Image: images/rico/16602.jpg
   ✅ Evaluation | Time: 1.99s | Reply: '$70'

Processing Task: 16603 | Image: images/rico/16603.jpg
   ✅ Evaluation | Time: 2.54s | Reply: 'Grace'

Processing Task: 16605 | Image: images/rico/16605.jpg
   ❌ Evaluation | Time: 1.88s | Reply: 'Facebook account'

Processing Task: 16608 | Image: images/rico/16608.jpg
   ✅ Evaluation | Time: 4.82s | Reply: 'appcrawler6@gmail.com'
🎯 Limit of 4 reached for openrouter/healer-alpha.


### Ngrok

In [11]:
def encode_image_from_bytes(byte_data):
    """Converts raw bytes to a base64 encoded string after resizing to 720p."""
    try:
        img = Image.open(BytesIO(byte_data))
        # Resize to 720p (maintaining aspect ratio)
        img.thumbnail((1280, 720))
        
        buffered = BytesIO()
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        img.save(buffered, format="JPEG", quality=85)
        base64_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding/resizing image: {e}")
        return None

def query_vision_model(prompt, base64_image, model_name, max_retries=MAX_RETRIES):
    """Sends the image and prompt directly via the OpenAI package to Ngrok backend."""
    strict_prompt = f"""{prompt}

You are an expert visual analyst. Answer the user's question using only the provided image and text.

CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:
1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler (e.g., never write "The answer is" or "Based on the image").
"""
    
    messages = [
         {
            "role": "user",
            "content": [
                {"type": "text", "text": strict_prompt},
                {"type": "image_url", "image_url": {"url": base64_image}}
            ]
        }
    ]
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_name,
                messages=messages,
                stream=False,
                temperature=0.1
            )
            result_text = response.choices[0].message.content.strip()
            duration = time.time() - start_time
            return result_text, duration
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_screenqa(ans, gt):
    """Evaluates if the model answer matches the ground truth with high tolerance for formatting."""
    ans_str = str(ans).lower()
    gt_str = str(gt).lower()
    
    # 1. Strip currencies and commas specifically to handle number formatting (e.g., $1,000 -> 1000)
    ans_str = re.sub(r'[\$\£\€\,]', '', ans_str)
    gt_str = re.sub(r'[\$\£\€\,]', '', gt_str)
    
    # 2. Strip ALL remaining non-alphanumeric characters (spaces, periods, colons, quotes)
    ans_clean = re.sub(r'[^\w]', '', ans_str)
    gt_clean = re.sub(r'[^\w]', '', gt_str)
    
    if not ans_clean or not gt_clean:
        return False
        
    # 3. Exact match
    if ans_clean == gt_clean:
        return True
        
    # 4. Ground truth is contained within the model's (potentially verbose) answer
    if gt_clean in ans_clean:
        return True
        
    # 5. Model's answer is contained within the ground truth 
    # (Requires length >= 2 to prevent single digits like "1" matching "100" as a false positive)
    if len(ans_clean) >= 2 and ans_clean in gt_clean:
        return True
        
    return False

def run_evaluation(df, limit=None):
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Starting ScreenQA NGROK inference using {model_name}...")
        
        safe_model_name = model_name.replace("/", "_").replace(":", "_")
        
        results_dir = Path("ScreenQAResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"screenqa_ngrok_{safe_model_name}_results.csv"
        
        processed_tasks = set()
        file_exists = output_csv.is_file()
        
        if file_exists:
            try:
                existing_df = pd.read_csv(output_csv)
                if not existing_df.empty and 'id' in existing_df.columns:
                    processed_tasks = set(existing_df['id'].astype(str))
                    print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
            except Exception as e:
                print(f"⚠️ Could not read existing CSV for progress tracking: {e}")
                
        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break
                
            task_id = str(row['screen_id'])
            if task_id in processed_tasks:
                continue
                
            image_name = str(row.get('file_name', 'N/A'))
            query = str(row['question'])
            
            # --- Safely extract 'full_answer' from the array of dicts ---
            raw_answers = row.get('answers', row.get('ground_truth', row.get('full_answer', [])))
            try:
                ground_truth = str(raw_answers[0]['full_answer']).strip()
            except (IndexError, KeyError, TypeError):
                ground_truth = str(raw_answers).strip()
            
            print(f"\nProcessing Task: {task_id} | Image: {image_name}")
            
            img_data = None
            if 'image' in row and isinstance(row['image'], dict) and 'bytes' in row['image']:
                img_data = row['image']['bytes']
            else:
                img_data = row.get('bytes') 
            
            base64_img = None
            if img_data:
                base64_img = encode_image_from_bytes(img_data)
            
            if not base64_img:
                print("   ⚠️ Skipping due to image parsing error.")
                continue
                
            model_answer, duration = query_vision_model(query, base64_img, model_name)
            evaluation_passed = evaluate_screenqa(model_answer, ground_truth)
            
            # Save to CSV
            result_df = pd.DataFrame([{ 
                'id': task_id,
                'image' : image_name,
                'question': query,
                'ground_truth': ground_truth,
                'model_answer': model_answer,
                'evaluation_passed': evaluation_passed,
                'run_time': round(duration, 3),
            }])
            
            result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
            file_exists = True
            
            processed_tasks.add(task_id)
            count += 1
            
            eval_mark = "✅" if evaluation_passed else "❌"
            print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
            time.sleep(3) # API rate limit pause

if 'df' not in globals():
    print("🔄 Dataframe 'df' not found. Loading ScreenQA dataset...")
    df = pq.read_table(dataset_path).to_pandas()

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_evaluation(df, limit=4)
else:
    print("❌ ‘df’ memory object missing. Please run the data loading cell first.")


🚀 Starting ScreenQA NGROK inference using qwen3-vl:4b...
🔄 Found existing CSV! Resuming... Skipping 2 already processed tasks.

Processing Task: 16605 | Image: images/rico/16605.jpg
   ✅ Evaluation | Time: 20.50s | Reply: 'Facebook and email'

Processing Task: 16608 | Image: images/rico/16608.jpg
   ✅ Evaluation | Time: 2.42s | Reply: 'appcrawler6@gmail.com'

Processing Task: 16609 | Image: images/rico/16609.jpg
   ✅ Evaluation | Time: 2.00s | Reply: 'Free'

Processing Task: 16610 | Image: images/rico/16610.jpg
   ❌ Evaluation | Time: 3.16s | Reply: 'Agreed'
🎯 Limit of 4 reached for qwen3-vl:4b.

🚀 Starting ScreenQA NGROK inference using llava:7b...

Processing Task: 16602 | Image: images/rico/16602.jpg
   ❌ Evaluation | Time: 5.49s | Reply: '$700'

Processing Task: 16603 | Image: images/rico/16603.jpg
   ❌ Evaluation | Time: 2.37s | Reply: 'The user name is "Dash".'

Processing Task: 16605 | Image: images/rico/16605.jpg
   ❌ Evaluation | Time: 3.50s | Reply: 'The image shows a screen

# MTVQA Evaluation

### Setup

In [6]:
# --- Data Loading ---
dataset_path = MTVQA_PATH

if dataset_path.exists():
    df = pd.read_parquet(dataset_path)
    print(f"✅ Loaded {len(df)} rows.")
else:
    print(f"❌ Dataset not found at {dataset_path}. Please check the path.")

✅ Loaded 818 rows.


### Openrouter

In [ ]:
import ast

def encode_image_from_bytes(byte_data):
    """Converts raw bytes to a base64 encoded string."""
    try:
        img = Image.open(BytesIO(byte_data))
        
        buffered = BytesIO()
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        img.save(buffered, format="JPEG", quality=85)
        base64_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"\u274c Error encoding/resizing image: {e}")
        return None

def query_vision_model_mtvqa(prompt, base64_image, MODEL_NAME, max_retries=MAX_RETRIES):
    """Sends the image and prompt directly to the OpenRouter API."""
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json"
    }
    
    strict_prompt = (
        f"{prompt}\n\n"
        "You are an expert multilingual visual analyst. Answer the user's question using only the provided image and text.\n\n"
        "CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:\n"
        "1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler "
        "(e.g., never write \"The answer is\" or \"Based on the image\").\n"
        "2. IMPORTANT: You MUST answer in the EXACT SAME LANGUAGE as the question. "
        "If the question is in Arabic, answer in Arabic. If in Chinese, answer in Chinese, etc.\n"
    )
    
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": strict_prompt},
                    {"type": "image_url", "image_url": {"url": base64_image}}
                ]
            }
        ],
        "temperature": 0.1
    }
    
    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = requests.post(
                url="https://openrouter.ai/api/v1/chat/completions",
                headers=headers,
                json=payload,
                timeout=120
            )
            response.raise_for_status()
            result_text = response.json()['choices'][0]['message']['content'].strip()
            duration = time.time() - start_time
            return result_text, duration
        except Exception as e:
            print(f"   \u274c Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")
            
        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))
            
    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_mtvqa(ans, gt):
    """Evaluates if the model answer matches the ground truth (exact or substring match)."""
    ans_clean = str(ans).strip()
    gt_clean = str(gt).strip()
    
    if not ans_clean or not gt_clean:
        return False
    
    # Exact match
    if ans_clean == gt_clean:
        return True
    
    # Ground truth contained within model answer
    if gt_clean in ans_clean:
        return True
    
    # Model answer contained within ground truth (min length 2)
    if len(ans_clean) >= 2 and ans_clean in gt_clean:
        return True
    
    return False

def run_mtvqa_evaluation(df, limit=None):
    print(f"\n\U0001f680 Starting MTVQA OPENROUTER inference using {MODEL_NAME}...")
    
    safe_MODEL_NAME = MODEL_NAME.replace("/", "_").replace(":", "_")
    
    results_dir = Path("MTVQAResults")
    results_dir.mkdir(parents=True, exist_ok=True)
    output_csv = results_dir / f"mtvqa_openrouter_{safe_MODEL_NAME}_results.csv"
    
    processed_tasks = set()
    file_exists = output_csv.is_file()
    
    if file_exists:
        try:
            existing_df = pd.read_csv(output_csv)
            if not existing_df.empty and 'id' in existing_df.columns:
                processed_tasks = set(existing_df['id'].astype(str))
                print(f"\U0001f504 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
        except Exception as e:
            print(f"\u26a0\ufe0f Could not read existing CSV for progress tracking: {e}")
    
    count = 0
    for index, row in df.iterrows():
        if limit and count >= limit:
            print(f"\U0001f3af Limit of {limit} reached for {MODEL_NAME}.")
            break
        
        task_id = str(row['id'])
        if task_id in processed_tasks:
            continue
        
        # --- Parse qa_pairs to extract question and ground truth answer ---
        try:
            qa_data = ast.literal_eval(row['qa_pairs']) if isinstance(row['qa_pairs'], str) else row['qa_pairs']
            query = str(qa_data[0]['question']).strip()
            ground_truth = str(qa_data[0]['answer']).strip()
        except Exception as e:
            print(f"   \u26a0\ufe0f Skipping task {task_id}: could not parse qa_pairs \u2014 {e}")
            continue
        
        lang = str(row.get('lang', 'N/A'))
        
        print(f"\nProcessing Task: {task_id} | Lang: {lang} | Q: {query[:60]}")
        
        # --- Extract image bytes ---
        img_data = None
        if 'image' in row and isinstance(row['image'], dict) and 'bytes' in row['image']:
            img_data = row['image']['bytes']
        else:
            img_data = row.get('bytes')
        
        base64_img = None
        if img_data:
            base64_img = encode_image_from_bytes(img_data)
        
        if not base64_img:
            print("   \u26a0\ufe0f Skipping due to image parsing error.")
            continue
        
        model_answer, duration = query_vision_model_mtvqa(query, base64_img, MODEL_NAME)
        evaluation_passed = evaluate_mtvqa(model_answer, ground_truth)
        
        # Save to CSV
        result_df = pd.DataFrame([{
            'id': task_id,
            'lang': lang,
            'question': query,
            'ground_truth': ground_truth,
            'model_answer': model_answer,
            'evaluation_passed': evaluation_passed,
            'run_time': round(duration, 3),
        }])
        
        result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
        file_exists = True
        
        processed_tasks.add(task_id)
        count += 1
        
        eval_mark = "\u2705" if evaluation_passed else "\u274c"
        print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
        time.sleep(3)  # API rate limit pause

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_mtvqa_evaluation(df, limit=2)
else:
    print("\u274c 'df' memory object missing. Please run the setup cell first.")


### Ollama (Local)

In [ ]:
import ast

# --- Local Ollama client (bypasses Ngrok, hits Ollama directly on this machine) ---
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=180.0
)

def encode_image_from_bytes_mtvqa(byte_data):
    """Converts raw bytes to a base64 encoded string after resizing to 720p."""
    try:
        img = Image.open(BytesIO(byte_data))
        img.thumbnail((1280, 720))
        buffered = BytesIO()
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        img.save(buffered, format="JPEG", quality=85)
        base64_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding/resizing image: {e}")
        return None

def query_vision_model_mtvqa_ollama(prompt, base64_image, model_name, max_retries=MAX_RETRIES):
    """Sends the image and prompt via the OpenAI client directly to local Ollama."""
    strict_prompt = (
        f"{prompt}\n\n"
        "You are an expert multilingual visual analyst. Answer the user's question using only the provided image and text.\n\n"
        "CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:\n"
        "1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler "
        "(e.g., never write \"The answer is\" or \"Based on the image\").\n"
        "2. IMPORTANT: You MUST answer in the EXACT SAME LANGUAGE as the question. "
        "If the question is in Arabic, answer in Arabic. If in Chinese, answer in Chinese, etc.\n"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": strict_prompt},
                {"type": "image_url", "image_url": {"url": base64_image}}
            ]
        }
    ]

    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = ollama_client.chat.completions.create(
                model=model_name,
                messages=messages,
                stream=False,
                temperature=0.1
            )
            result_text = response.choices[0].message.content.strip()
            duration = time.time() - start_time
            return result_text, duration
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")

        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))

    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_mtvqa_ollama(ans, gt):
    """Evaluates if the model answer matches the ground truth (exact or substring match)."""
    ans_clean = str(ans).strip()
    gt_clean = str(gt).strip()

    if not ans_clean or not gt_clean:
        return False
    if ans_clean == gt_clean:
        return True
    if gt_clean in ans_clean:
        return True
    if len(ans_clean) >= 2 and ans_clean in gt_clean:
        return True
    return False

def run_mtvqa_ollama_evaluation(df, limit=None):
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Starting MTVQA Ollama inference using {model_name}...")

        safe_model_name = model_name.replace("/", "_").replace(":", "_")

        results_dir = Path("MTVQAResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"mtvqa_ollama_{safe_model_name}_results.csv"

        processed_tasks = set()
        file_exists = output_csv.is_file()

        if file_exists:
            try:
                existing_df = pd.read_csv(output_csv)
                if not existing_df.empty and 'id' in existing_df.columns:
                    processed_tasks = set(existing_df['id'].astype(str))
                    print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
            except Exception as e:
                print(f"⚠️ Could not read existing CSV for progress tracking: {e}")

        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            task_id = str(row['id'])
            if task_id in processed_tasks:
                continue

            # --- Parse qa_pairs to extract question and ground truth answer ---
            try:
                qa_data = ast.literal_eval(row['qa_pairs']) if isinstance(row['qa_pairs'], str) else row['qa_pairs']
                query = str(qa_data[0]['question']).strip()
                ground_truth = str(qa_data[0]['answer']).strip()
            except Exception as e:
                print(f"   ⚠️ Skipping task {task_id}: could not parse qa_pairs — {e}")
                continue

            lang = str(row.get('lang', 'N/A'))

            print(f"\nProcessing Task: {task_id} | Lang: {lang} | Q: {query[:60]}")

            # --- Extract image bytes ---
            img_data = None
            if 'image' in row and isinstance(row['image'], dict) and 'bytes' in row['image']:
                img_data = row['image']['bytes']
            else:
                img_data = row.get('bytes')

            base64_img = None
            if img_data:
                base64_img = encode_image_from_bytes_mtvqa(img_data)

            if not base64_img:
                print("   ⚠️ Skipping due to image parsing error.")
                continue

            model_answer, duration = query_vision_model_mtvqa_ollama(query, base64_img, model_name)
            evaluation_passed = evaluate_mtvqa_ollama(model_answer, ground_truth)

            # Save to CSV
            result_df = pd.DataFrame([{
                'id': task_id,
                'lang': lang,
                'question': query,
                'ground_truth': ground_truth,
                'model_answer': model_answer,
                'evaluation_passed': evaluation_passed,
                'run_time': round(duration, 3),
            }])

            result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
            file_exists = True

            processed_tasks.add(task_id)
            count += 1

            eval_mark = "✅" if evaluation_passed else "❌"
            print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
            time.sleep(3)  # Pause between requests

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_mtvqa_ollama_evaluation(df, limit=2)
else:
    print("❌ 'df' memory object missing. Please run the setup cell first.")


# PEARL Evaluation

### Setup

In [25]:
# --- Data Loading ---
dataset_path = PEARL_PATH

if dataset_path.exists():
    df = pd.read_parquet(dataset_path)
    print(f"✅ Loaded {len(df)} rows.")
else:
    print(f"❌ Dataset not found at {dataset_path}. Please check the path.")

✅ Loaded 4832 rows.


### Ollama

In [26]:
# --- Local Ollama client ---
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
    timeout=180.0
)

def encode_image_from_bytes_pearl(byte_data):
    """Converts raw bytes to a base64 encoded string"""
    try:
        img = Image.open(BytesIO(byte_data))
        buffered = BytesIO()
        if img.mode in ("RGBA", "P"):
            img = img.convert("RGB")
        img.save(buffered, format="JPEG", quality=85)
        base64_str = base64.b64encode(buffered.getvalue()).decode('utf-8')
        return f"data:image/jpeg;base64,{base64_str}"
    except Exception as e:
        print(f"❌ Error encoding/resizing image: {e}")
        return None

def query_vision_model_pearl(prompt, base64_image, model_name, max_retries=MAX_RETRIES):
    """Sends the image and prompt via the OpenAI client directly to local Ollama."""
    strict_prompt = (
        f"{prompt}\n\n"
        "You are an expert visual analyst. Answer the user's question using only the provided image and text.\n\n"
        "CRITICAL INSTRUCTIONS FOR AUTOMATED EVALUATION:\n"
        "1. Output ONLY the final answer. Provide zero explanations, reasoning, or conversational filler "
        "(e.g., never write \"The answer is\" or \"Based on the image\").\n"
        "2. IMPORTANT: You MUST answer in the EXACT SAME LANGUAGE as the question. "
        "If the question is in Arabic, answer in Arabic. If in English, answer in English, etc.\n"
        "If it is a true and false question answer TRUE or FALSE, if it a multiple choice question give me the answer itself and NOT the letter. \n"
    )

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": strict_prompt},
                {"type": "image_url", "image_url": {"url": base64_image}}
            ]
        }
    ]

    start_time = time.time()
    for attempt in range(max_retries):
        try:
            response = ollama_client.chat.completions.create(
                model=model_name,
                messages=messages,
                stream=False,
                temperature=0.1
            )
            result_text = response.choices[0].message.content.strip()
            duration = time.time() - start_time
            return result_text, duration
        except Exception as e:
            print(f"   ❌ Attempt {attempt + 1}/{max_retries}: API Request failed: {e}")

        if attempt < max_retries - 1:
            time.sleep(5 * (attempt + 1))

    duration = time.time() - start_time
    return "ERROR_MAX_RETRIES_REACHED", duration

def evaluate_pearl(ans, gt):
    """Evaluates if the model answer matches the ground truth (exact or substring match)."""
    ans_clean = str(ans).strip()
    gt_clean = str(gt).strip()

    if not ans_clean or not gt_clean:
        return False
    if ans_clean == gt_clean:
        return True
    if gt_clean in ans_clean:
        return True
    if len(ans_clean) >= 2 and ans_clean in gt_clean:
        return True
    return False

def run_pearl_ollama_evaluation(df, limit=None):
    for model_name in MODELS_TO_TEST:
        print(f"\n🚀 Starting PEARL Ollama inference using {model_name}...")

        safe_model_name = model_name.replace("/", "_").replace(":", "_")

        results_dir = Path("PEARLResults")
        results_dir.mkdir(parents=True, exist_ok=True)
        output_csv = results_dir / f"pearl_ollama_{safe_model_name}_results.csv"

        processed_tasks = set()
        file_exists = output_csv.is_file()

        if file_exists:
            try:
                existing_df = pd.read_csv(output_csv)
                if not existing_df.empty and 'index' in existing_df.columns:
                    processed_tasks = set(existing_df['index'].astype(str))
                    print(f"🔄 Found existing CSV! Resuming... Skipping {len(processed_tasks)} already processed tasks.")
            except Exception as e:
                print(f"⚠️ Could not read existing CSV for progress tracking: {e}")

        count = 0
        for index, row in df.iterrows():
            if limit and count >= limit:
                print(f"🎯 Limit of {limit} reached for {model_name}.")
                break

            # Use the dataframe index as the unique index
            index = str(index)
            if index in processed_tasks:
                continue

            query        = str(row['question']).strip()
            ground_truth = str(row['answer']).strip()
            category     = str(row.get('category', 'N/A'))
            image_id     = str(row.get('image_id', 'N/A'))

            print(f"\nProcessing Task: {index} | image_id: {image_id} | category: {category}")

            # --- Extract image bytes ---
            img_data = None
            if 'image' in row and isinstance(row['image'], dict) and 'bytes' in row['image']:
                img_data = row['image']['bytes']
            else:
                img_data = row.get('bytes')

            base64_img = None
            if img_data:
                base64_img = encode_image_from_bytes_pearl(img_data)

            if not base64_img:
                print("   ⚠️ Skipping due to image parsing error.")
                continue

            model_answer, duration = query_vision_model_pearl(query, base64_img, model_name)
            evaluation_passed = evaluate_pearl(model_answer, ground_truth)

            # Save to CSV
            result_df = pd.DataFrame([{
                'index'   : index,
                'image_id'         : image_id,
                'category'         : category,
                'question'         : query,
                'ground_truth'     : ground_truth,
                'model_answer'     : model_answer,
                'evaluation_passed': evaluation_passed,
                'run_time'         : round(duration, 3),
            }])

            result_df.to_csv(output_csv, mode='a', header=not file_exists, index=False)
            file_exists = True

            processed_tasks.add(index)
            count += 1

            eval_mark = "✅" if evaluation_passed else "❌"
            print(f"   {eval_mark} Evaluation | Time: {duration:.2f}s | Reply: '{model_answer}'")
            time.sleep(3)  # Pause between requests

if 'df' in globals():
    # Uncomment the limit parameter to test on a subset first
    run_pearl_ollama_evaluation(df, limit=4)
else:
    print("❌ 'df' memory object missing. Please run the setup cell first.")



🚀 Starting PEARL Ollama inference using qwen3-vl:4b...

Processing Task: 0 | image_id: 131625 | category: LandMarks
   ❌ Evaluation | Time: 83.47s | Reply: 'السراي الحمراء'

Processing Task: 1 | image_id: 32236 | category: Architecture
   ✅ Evaluation | Time: 25.43s | Reply: 'TRUE'

Processing Task: 2 | image_id: 78328 | category: Architecture
   ❌ Evaluation | Time: 17.94s | Reply: 'الصورة تُظهر نقوشًا على حجر مُحفوظة تُبرز جنودًا يُقاتلون، مما يعكس أهمية الحرب في الثقافة الآشورية عبر تمثيلها لمشاهد القتال والصراعات.'

Processing Task: 3 | image_id: 38927.png | category: Clothes
   ❌ Evaluation | Time: 140.06s | Reply: 'FALSE'
🎯 Limit of 4 reached for qwen3-vl:4b.


# Results Evaluation

In [ ]:
def summarize_results():
    results_folders = ["MathVisionResults", "ChartQAResults", "ScreenQAResults", "TurtleBenchResults", "MTVQAResults", "PEARLResults"]
    
    # Print Header
    print("{:<60} | {:<10}".format("Model Name", "Accuracy"))
    print("-" * 75)
    
    for folder in results_folders:
        folder_path = Path(folder)
        if not folder_path.exists():
            continue
            
        for csv_file in folder_path.glob("*.csv"):
            try:
                df = pd.read_csv(csv_file)
                if "evaluation_passed" not in df.columns:
                    continue
                
                # Calculate accuracy
                total_rows = len(df)
                if total_rows == 0:
                    continue
                    
                # Ensure evaluation_passed is numeric for summing
                true_answers = df["evaluation_passed"].astype(int).sum()
                accuracy = (true_answers / total_rows) * 100
                
                # Clean up filename for display
                name = csv_file.stem.replace("_ngrok", "").replace("_results", "")
                
                print("{:<60} | {:>8.2f}%".format(name, accuracy))
            except Exception as e:
                print("⚠️ Error processing {}: {}".format(csv_file.name, e))

summarize_results()

Model Name                                                   | Accuracy  
---------------------------------------------------------------------------
mathvision_llava_7b                                          |    50.00%
mathvision_qwen3-vl_8b                                       |   100.00%
chartqa_qwen3-vl_4b                                          |   100.00%
chartqa_openrouter_openrouter_healer-alpha                   |   100.00%
chartqa_llava_7b                                             |     0.00%
screenqa_llava_7b                                            |     0.00%
screenqa_openrouter_openrouter_healer-alpha                  |    75.00%
screenqa_qwen3-vl_4b                                         |    83.33%
TurtleBench_openrouter_healer-alpha                          |     0.00%
mtvqa_ollama_qwen3-vl_4b                                     |     0.00%
